In [1]:
import os
from dotenv import load_dotenv
from typing import List, Optional, TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import load_prompt
from langchain_groq import ChatGroq
from langchain_community.graphs import Neo4jGraph
from langgraph.graph import StateGraph, END
from langchain_community.callbacks.manager import get_openai_callback
from langchain_core.output_parsers import JsonOutputParser

from pydantic import BaseModel, Field

# --- 1. CONFIGURATION & INITIALIZATION (from your notebook) ---
# Assuming variables are already loaded from .env as per your notebook

load_dotenv()

NEO4J_URL = os.getenv("NEO4J_URL")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
groq_api_key = os.getenv("GROQ_API_KEY")

graph = Neo4jGraph(
    url=NEO4J_URL, 
    username=NEO4J_USER, 
    password=NEO4J_PASSWORD, 
    database=NEO4J_USER,
    refresh_schema=True 
)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
smaller_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

cypher_prompt = load_prompt("cypher-few-shot.yaml")
qa_prompt_template = load_prompt("qa-prompt.yaml")


/var/folders/r9/wng9hy551n370k7tjvryjqbw0000gn/T/ipykernel_6680/2042201289.py:25: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [2]:
class ScopeAnalysis(BaseModel):
    in_scope: bool = Field(description="Is the question about crude oil trade or energy logistics?")
    reason: str = Field(description="Brief explanation for the scope decision.")

parser = JsonOutputParser(pydantic_object=ScopeAnalysis)

class CypherResponse(BaseModel):
    cypher_query: str = Field(description="The executable Neo4j Cypher query.")
    # complexity_score: int = Field(description="1-10 score of query complexity.")
    # reasoning: str = Field(description="Brief explanation of the query logic.")

cypher_parser = JsonOutputParser(pydantic_object=CypherResponse)

In [3]:

# --- 2. STATE DEFINITION ---
class AgentState(TypedDict):
    question: str
    cypher: Optional[str]
    error: Optional[str]
    results: List
    iterations: int
    final_response: Optional[str]
    in_scope: bool
    # Ensure these three are present
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

# --- 3. NODE DEFINITIONS ---

# def token_logger_node(state: AgentState):
#     """Prints a detailed breakdown of the token economy for the request."""
#     print(f"\n" + "="*40)
#     print(f"📊 SENTINEL TOKEN AUDIT")
#     print(f"="*40)
    
#     # We use .get() to avoid errors if a node was skipped (like an out-of-scope query)
#     print(f"Prompt Tokens:     {state.get('prompt_tokens', 0)}")
#     print(f"Completion Tokens: {state.get('completion_tokens', 0)}")
#     print(f"Total Tokens:      {state.get('total_tokens', 0)}")
#     print(f"Workflow Iterations: {state.get('iterations', 0)}")
    
#     # Calculate utilization against the 128k context window
#     usage_pct = (state.get('total_tokens', 0) / 128000) * 100
#     print(f"Context Window Usage: {usage_pct:.2f}%")
#     print("="*40 + "\n")
    
#     return state # Pass-through to END

SCOPE_SYSTEM_PROMPT = "You are the Gatekeeper for an oil trade system. \
If a query is about trade, imports, or exports, assume it refers to crude oil (HS 2709) unless another commodity is named. \
Output ONLY JSON per {format_instructions}. DO NOT answer the user."

def check_scope_node(state: AgentState):
    # Inject format instructions into the prompt
    system_msg = SCOPE_SYSTEM_PROMPT.format(
        format_instructions=parser.get_format_instructions()
    )
    
    with get_openai_callback() as cb:
        # Use regular .invoke instead of .with_structured_output
        response = smaller_llm.invoke([
            SystemMessage(content=system_msg),
            HumanMessage(content=state['question'])
        ])
        
        # Parse the raw JSON string into a dictionary
        analysis = parser.parse(response.content)
    
    return {
        "in_scope": analysis['in_scope'],
        "prompt_tokens": cb.prompt_tokens,
        "completion_tokens": cb.completion_tokens,
        "total_tokens": cb.total_tokens,
        "iterations": 0, "error": None
    }

def generate_cypher_node(state: AgentState):
    """Generates structured Cypher using JsonOutputParser for stability."""
    error_msg = f"\n\nPrevious attempt failed: {state['error']}. Fix the query." if state['error'] else ""
    
    # 3. Inject format instructions into your YAML-loaded prompt
    format_instructions = cypher_parser.get_format_instructions()
    
    formatted_prompt = cypher_prompt.format(
        question=state["question"], 
        schema=graph.schema
    ) + f"\n\nIMPORTANT: You must output your response as a JSON object strictly following this format:\n{format_instructions}" + error_msg
    
    with get_openai_callback() as cb:
        # Use standard .invoke
        response = llm.invoke(formatted_prompt)
        
        # 4. Parse the raw JSON string
        # This handles cases where the LLM might include markdown backticks
        clean_content = response.content.replace("```json", "").replace("```", "").strip()
        structured_data = cypher_parser.parse(clean_content)
        
    return {
        "cypher": structured_data['cypher_query'], 
        "iterations": state["iterations"] + 1,
        "prompt_tokens": state.get("prompt_tokens", 0) + cb.prompt_tokens,
        "completion_tokens": state.get("completion_tokens", 0) + cb.completion_tokens,
        "total_tokens": state.get("total_tokens", 0) + cb.total_tokens
    }

def execute_cypher_node(state: AgentState):
    """Attempts execution on Neo4j."""
    try:
        # Clean the Cypher string (remove markdown if the LLM added it)
        clean_cypher = state["cypher"].replace("```cypher", "").replace("```", "").strip()
        results = graph.query(clean_cypher)
        return {"results": results, "error": None}
    except Exception as e:
        return {"error": str(e), "results": []}

def responder_node(state: AgentState):
    """Generates the final natural language answer or fallback."""
    # 1. Fallback for Out-of-Scope
    if not state["in_scope"]:
        return {"final_response": "I am sorry, but your request is outside the objectives of this oil trade intelligence project. I can only assist with queries related to crude oil imports and global energy logistics."}

    # 2. Fallback for Repeated Cypher Errors (Humble Refusal)
    if state["error"] and state["iterations"] >= 2:
        return {"final_response": "I cannot answer this query right now due to technical difficulties with the database query."}

    # 3. Handle Empty Results
    if not state["results"]:
        return {"final_response": "No matching oil trade data was found for your specific query records."}

    # 4. Standard Response using your YAML QA Prompt logic
    # We use the QA prompt template loaded from your YAML to maintain formatting rules
    full_qa_prompt = qa_prompt_template.format(
        question=state["question"],
        context=str(state["results"])
    )
    
    with get_openai_callback() as cb:
            summary = llm.invoke(full_qa_prompt)
            
    return {
            "final_response": summary.content,
            "prompt_tokens": state.get("prompt_tokens", 0) + cb.prompt_tokens,
            "completion_tokens": state.get("completion_tokens", 0) + cb.completion_tokens,
            "total_tokens": state.get("total_tokens", 0) + cb.total_tokens
        }

In [4]:
# --- 4. GRAPH CONSTRUCTION ---
workflow = StateGraph(AgentState)

# Add all nodes, including the new Logger
workflow.add_node("check_scope", check_scope_node)
workflow.add_node("generate_cypher", generate_cypher_node)
workflow.add_node("execute_cypher", execute_cypher_node)
workflow.add_node("responder", responder_node)
# workflow.add_node("token_logger", token_logger_node) # Node added

workflow.set_entry_point("check_scope")

# Branching logic
workflow.add_conditional_edges(
    "check_scope", 
    lambda x: "generate_cypher" if x["in_scope"] else "responder"
)

workflow.add_edge("generate_cypher", "execute_cypher")

# Retry logic: Fixes errors up to 2 times
def retry_logic(state):
    if state["error"] and state["iterations"] < 2:
        return "generate_cypher"
    return "responder"

workflow.add_conditional_edges("execute_cypher", retry_logic)

# IMPORTANT: Ensure responder flows into the logger before ending
# workflow.add_edge("responder", "token_logger")
# workflow.add_edge("token_logger", END)

# Compile the app
app = workflow.compile()

# --- 5. EXECUTION WRAPPER ---
import time

def ask_sentinel(question: str):
    inputs = {"question": question}
    config = {"recursion_limit": 10}
    final_response = None
    
    total_start = time.time()
    last_node_time = total_start # Mark the start of the first node

    print(f"🚀 Starting Sentinel Analysis: {question}\n")

    # Use app.stream to catch node completion in real-time
    for output in app.stream(inputs, config=config):
        # Calculate how long the node took to finish
        current_time = time.time()
        node_duration = current_time - last_node_time
        
        for node_name, state_update in output.items():
            print(f"[{node_name.upper()}] (⏱️ {node_duration:.2f}s)")
            
            # Visibility: Scope Check
            if "in_scope" in state_update:
                status = "✅ IN-SCOPE" if state_update['in_scope'] else "❌ OUT-OF-SCOPE"
                print(f"   Status: {status}")
            
            # Visibility: Cypher Generation
            if "cypher" in state_update:
                print(f"   Cypher: {state_update['cypher']}")
            
            # Visibility: Execution Results
            if "results" in state_update:
                print(f"   Query Results: {len(state_update['results'])} records found")
            
            # Visibility: Final Answer
            if "final_response" in state_update:
                final_response = state_update["final_response"]
            
        # Reset the timer for the next node in the sequence
        last_node_time = time.time()

    total_duration = time.time() - total_start
    print(f"\n✅ Analysis Complete")
    print(f"⏱️ Total Response Time: {total_duration:.2f}s")
    
    return final_response

In [5]:
print(ask_sentinel("Top 10 oil importers of the world"))

🚀 Starting Sentinel Analysis: Top 10 oil importers of the world

[CHECK_SCOPE] (⏱️ 0.29s)
   Status: ✅ IN-SCOPE
[GENERATE_CYPHER] (⏱️ 1.60s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer.name AS importer_name, ROUND(sum(f.barrels) / 1000000, 2) AS total_imported_volume_millions
RETURN importer_name, total_imported_volume_millions
ORDER BY total_imported_volume_millions DESC
LIMIT 10
[EXECUTE_CYPHER] (⏱️ 0.21s)
   Query Results: 10 records found
[RESPONDER] (⏱️ 1.11s)

✅ Analysis Complete
⏱️ Total Response Time: 3.21s
Based on total trade for 2023...  
China: 4024.46 million barrels  
USA: 2313.18 million barrels  
India: 1766.49 million barrels  
Rep. of Korea: 1009.12 million barrels  
Japan: 860.3 million barrels  
Netherlands: 584.11 million barrels  
Germany: 581.76 m

In [6]:
print(ask_sentinel("Largest exporter of crude oil in South East Asia?"))

🚀 Starting Sentinel Analysis: Largest exporter of crude oil in South East Asia?

[CHECK_SCOPE] (⏱️ 0.26s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.95s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {region: 'South East Asia'})
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels)/1000000, 2) AS total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 1
[EXECUTE_CYPHER] (⏱️ 0.20s)
   Query Results: 1 records found
[RESPONDER] (⏱️ 1.13s)

✅ Analysis Complete
⏱️ Total Response Time: 3.54s
Based on total trade for the most recent year, the largest exporter of crude oil in South East Asia is Malaysia with 595.5 million barrels.


In [6]:
res = ask_sentinel("Which country provided the most barrels to the USA in 202401?")
print(res)

🚀 Starting Sentinel Analysis: Which country provided the most barrels to the USA in 202401?

[CHECK_SCOPE]
Status: ✅ IN-SCOPE
Query Results: []
[GENERATE_CYPHER]
Cypher: MATCH (importer:Country {name: 'USA'})-[:IMPORTS_IN]->(tm:YearMonth {yearMonth: 202401})
MATCH (importer)-[f:IMPORTED_FROM {year_month: 202401}]->(exporter:Country)
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels)/1000000, 2) AS total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 5
[EXECUTE_CYPHER]
Query Results: [{'exporter': 'Canada', 'total_volume_millions': 126.42}, {'exporter': 'Mexico', 'total_volume_millions': 16.6}, {'exporter': 'Saudi Arabia', 'total_volume_millions': 9.27}, {'exporter': 'Colombia', 'total_volume_millions': 7.71}, {'exporter': 'Brazil', 'total_volume_millions': 7.71}]
[RESPONDER]
Based on total trade for Jan 2024, Canada provided the most barrels to the USA, with 126.42 million barrels.


In [7]:
res = ask_sentinel("How much barrels India imported in 202305?")
print(res)

🚀 Starting Sentinel Analysis: How much barrels India imported in 202305?

[CHECK_SCOPE]
Status: ✅ IN-SCOPE
Query Results: []
[GENERATE_CYPHER]
Cypher: MATCH (importer:Country {name: 'India'})-[:IMPORTS_IN]->(t:YearMonth {yearMonth: 202305})
MATCH (importer)-[f:IMPORTED_FROM {year_month: 202305}]->(exporter:Country)
RETURN ROUND(sum(f.barrels)/1000000, 2) AS total_volume_barrels_in_millions LIMIT 5
[EXECUTE_CYPHER]
Query Results: [{'total_volume_barrels_in_millions': 161.24}]
[RESPONDER]
Based on total trade for May 2023, India imported 161.24 million barrels.


In [7]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their number of barrels")
print(res)

🚀 Starting Sentinel Analysis: If Russians exports drop by 20%, which partner countries have the highest risk based on their number of barrels

[CHECK_SCOPE] (⏱️ 0.54s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.59s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {name: 'Russian Federation'})
WITH importer.name AS country, sum(f.barrels) AS total_barrels
WITH country, total_barrels, total_barrels * 0.2 AS risk_barrels
RETURN country,
       ROUND(total_barrels/1000000, 2) AS total_volume_millions,
       ROUND(risk_barrels/1000000, 2) AS potential_loss_millions
ORDER BY risk_barrels DESC
LIMIT 3
[EXECUTE_CYPHER] (⏱️ 0.10s)
   Query Results: 3 records found
[RESPONDER] (⏱️ 1.21s)

✅ Analysis Complete
⏱️ Total Response Time: 3.44s
Based on total trade fo

In [8]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their historical reliance?")
print(res)

🚀 Starting Sentinel Analysis: If Russians exports drop by 20%, which partner countries have the highest risk based on their historical reliance?

[CHECK_SCOPE] (⏱️ 0.55s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.95s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
WITH importer,
     russian_vol,
     total_vol,
     ROUND(russian_vol / 1000000, 2) AS russian_millions,
     ROUND((russian_vol / total_vol) * 100, 2) AS reliance_percent,
     ROUND(russian_vol * 0.2 / 1000000, 2) AS potential_loss_millions
ORDER BY reliance_percent DESC
LIMIT 3
RETURN importer.name AS country,
       rus

In [9]:
res = ask_sentinel("Which country is most reliable on Russia for their imports?")
print(res)

🚀 Starting Sentinel Analysis: Which country is most reliable on Russia for their imports?

[CHECK_SCOPE] (⏱️ 0.40s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.47s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
// Filter to months in the latest calendar year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
RETURN importer.name AS country,
       ROUND(russian_vol / 1000000, 2) AS barrels_from_russia_millions,
       ROUND((russian_vol / total_vol) * 100, 2) AS reliance_percent
ORDER BY reliance_percent DESC
LIMIT 1
[EXECUTE_CYPHER] (⏱️ 0.26s)
   Query Results: 1 records found
[RESPONDER] (⏱️ 1.04s)

✅ Analysis Complete
⏱️ Total Respon

In [10]:
res = ask_sentinel("Which country is most reliable on Russia for their imports for 2027?")
print(res)

🚀 Starting Sentinel Analysis: Which country is most reliable on Russia for their imports for 2027?

[CHECK_SCOPE] (⏱️ 0.35s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 2.02s)
   Cypher: MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth) WHERE tm.yearMonth / 100 = 2027 MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country) WITH importer, sum(f.barrels) AS total_vol, sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol WHERE total_vol > 0 RETURN importer.name AS country, ROUND(russian_vol/1000000,2) AS barrels_from_russia_millions, ROUND((russian_vol/total_vol)*100,2) AS reliance_percent ORDER BY reliance_percent DESC LIMIT 1
[EXECUTE_CYPHER] (⏱️ 0.13s)
   Query Results: 0 records found
[RESPONDER] (⏱️ 0.00s)

✅ Analysis Complete
⏱️ Total Response Time: 2.50s
No matching oil trade data was found for your specific query records.


In [14]:
res = ask_sentinel("Find instances where an importer is importing the same commodity from two different exporter at a price difference of >30%.")
print(res)

🚀 Starting Sentinel Analysis: Find instances where an importer is importing the same commodity from two different exporter at a price difference of >30%.

[CHECK_SCOPE] (⏱️ 0.50s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 5.94s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth)/100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth/100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer, tm.yearMonth AS ym, exporter.name AS exporterName, f.price_mt AS price
ORDER BY importer.name, ym, price
WITH importer, ym, collect({exporter:exporterName, price:price}) AS exporters
WHERE size(exporters) >= 2
UNWIND exporters AS e1
UNWIND exporters AS e2
WITH importer, ym, e1, e2
WHERE e1.exporter <> e2.exporter AND e1.price > 0 AND e2.price > 0
WITH importer, ym, e1, e2,
     CASE WHEN e1.price > e2.price THEN e1.price / e2.price ELSE e2.price / e1.price END AS ratio


In [15]:
res = ask_sentinel("Who is the largest exporter of oil?")
print(res)

🚀 Starting Sentinel Analysis: Who is the largest exporter of oil?

[CHECK_SCOPE] (⏱️ 0.33s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.93s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH exporter.name AS exporter, ROUND(sum(f.barrels) / 1000000, 2) AS total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 3
RETURN exporter, total_volume_millions
[EXECUTE_CYPHER] (⏱️ 0.27s)
   Query Results: 3 records found
[RESPONDER] (⏱️ 1.78s)

✅ Analysis Complete
⏱️ Total Response Time: 4.31s
Based on total trade for the latest year, the largest exporter of oil is Saudi Arabia with 2145.19 million barrels.


In [22]:
res = ask_sentinel("If there is a war in the middle east then which countries imports will be affected and by how much?\
Will India be affected?")
print(res)

🚀 Starting Sentinel Analysis: If there is a war in the middle east then which countries imports will be affected and by how much?Will India be affected?

[CHECK_SCOPE] (⏱️ 0.46s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 3.89s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
// Filter to months in the latest calendar year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
// Imports from exporters located in the Middle East region
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {region: 'Middle East'})
WITH importer, tm, sum(f.barrels) AS middle_vol
// Total imports for the same importer/month (all partners)
MATCH (importer)-[allF:IMPORTED_FROM {year_month: tm.yearMonth}]->(anyExporter:Country)
WITH importer, sum(middle_vol) AS middle_total, sum(allF.barrels) AS total_vol
WHERE middle_total > 0
RETURN importer.name AS country,
       ROUND(middle_tot

In [16]:
res = ask_sentinel("Largest importer of crude oil in South East Asia?")
print(res)

🚀 Starting Sentinel Analysis: Largest importer of crude oil in South East Asia?

[CHECK_SCOPE] (⏱️ 0.45s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 2.51s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year AND importer.region = 'South East Asia'
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer.name AS importer, ROUND(sum(f.barrels) / 1000000, 2) AS total_volume_millions
RETURN importer, total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 3
[EXECUTE_CYPHER] (⏱️ 0.16s)
   Query Results: 3 records found
[RESPONDER] (⏱️ 1.04s)

✅ Analysis Complete
⏱️ Total Response Time: 4.16s
Based on total trade for 2023, the largest importer of crude oil in South East Asia is Thailand with 402.75 million barrels.


In [17]:
res = ask_sentinel("How to bake French bread?")
print(res)

🚀 Starting Sentinel Analysis: How to bake French bread?

[CHECK_SCOPE] (⏱️ 0.35s)
   Status: ❌ OUT-OF-SCOPE
   Query Results: 0 records found
[RESPONDER] (⏱️ 0.00s)

✅ Analysis Complete
⏱️ Total Response Time: 0.35s
I am sorry, but your request is outside the objectives of this oil trade intelligence project. I can only assist with queries related to crude oil imports and global energy logistics.


In [27]:
res = ask_sentinel("If there is a conflict between Iran and Israel then which countries imports will be affected?")
print(res)

#The LLM does not inherently know that a conflict between Iran and Israel might lead to a blockade of the Strait of Hormuz,
#  which would affect all Middle Eastern oil (Saudi, UAE, Kuwait, etc.). 
#It only knows what is in the relationship lines.


# 10. GEOPOLITICAL IMPACT: If a question mentions a conflict or 'risk' in the Middle East (e.g., Iran, Israel, Red Sea):
        # a) First, query direct trade for those specific countries.
        # b) Second, automatically query the total imports from the broader region: 
        #    ['Saudi Arabia', 'United Arab Emirates', 'Iraq', 'Kuwait', 'Qatar', 'Oman'].
        # c) Return BOTH sets of data so the user can see the direct and regional exposure.

🚀 Starting Sentinel Analysis: If there is a conflict between Iran and Israel then which countries imports will be affected?

[CHECK_SCOPE] (⏱️ 1.86s)
   Status: ✅ IN-SCOPE
   Query Results: 0 records found
[GENERATE_CYPHER] (⏱️ 1.33s)
   Cypher: MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WHERE exporter.name IN ['Iran','Israel']
RETURN DISTINCT importer.name AS affected_importer
ORDER BY importer.name
LIMIT 5
[EXECUTE_CYPHER] (⏱️ 0.19s)
   Query Results: 1 records found
[RESPONDER] (⏱️ 1.19s)

✅ Analysis Complete
⏱️ Total Response Time: 4.58s
Based on total trade for 2023... The Netherlands imports will be affected.
